In [1]:
import pandas as pd
import os

# Define the core paths (since the notebook is inside 'notebooks/' folder, we go one step back '..')
BASE_DIR = '../data/raw/CrisisMMD_v2.0/'
ANNOTATION_DIR = os.path.join(BASE_DIR, 'crisismmd_datasplit_all', 'crisismmd_datasplit_all')

# 1. Let's see what TSV files we have
print("Available Annotation Files:")
files = [f for f in os.listdir(ANNOTATION_DIR) if f.endswith('.tsv')]
for f in files:
    print(f" - {f}")

# 2. Load the primary training dataset annotation
# Task: Humanitarian (Informative vs Not Informative)
train_tsv_path = os.path.join(ANNOTATION_DIR, 'task_humanitarian_text_img_train.tsv')

if os.path.exists(train_tsv_path):
    df = pd.read_csv(train_tsv_path, sep='\t')
    print(f"\n[+] Total Data Points in Train Set: {len(df)}")
    print("\n[+] Columns in the dataset:")
    print(df.columns.tolist())
    
    # Display the first 3 rows to inspect the data structure
    display(df.head(3))
else:
    print(f"[-] File not found: {train_tsv_path}. Check the exact filenames printed above.")

Available Annotation Files:
 - task_damage_text_img_dev.tsv
 - task_damage_text_img_test.tsv
 - task_damage_text_img_train.tsv
 - task_humanitarian_text_img_dev.tsv
 - task_humanitarian_text_img_test.tsv
 - task_humanitarian_text_img_train.tsv
 - task_informative_text_img_dev.tsv
 - task_informative_text_img_test.tsv
 - task_informative_text_img_train.tsv

[+] Total Data Points in Train Set: 13608

[+] Columns in the dataset:
['event_name', 'tweet_id', 'image_id', 'tweet_text', 'image', 'label', 'label_text', 'label_image', 'label_text_image']


,event_name,tweet_id,image_id,tweet_text,image,label,label_text,label_image,label_text_image
0,california_wildfires,917791291823591425,917791291823591425_1,RT @Cal_OES: PLS SHARE: Weâ€™re capturing wild...,data_image/california_wildfires/10_10_2017/917...,not_humanitarian,other_relevant_information,not_humanitarian,Negative
1,california_wildfires,917791291823591425,917791291823591425_0,RT @Cal_OES: PLS SHARE: Weâ€™re capturing wild...,data_image/california_wildfires/10_10_2017/917...,other_relevant_information,other_relevant_information,infrastructure_and_utility_damage,Negative
2,california_wildfires,917793137925459968,917793137925459968_0,RT @KAKEnews: California wildfires destroy mor...,data_image/california_wildfires/10_10_2017/917...,infrastructure_and_utility_damage,infrastructure_and_utility_damage,infrastructure_and_utility_damage,Positive


In [2]:
# 1. Check the exact distribution of disaster events
print("--- Event Distribution (What kind of disasters are here?) ---")
display(df['event_name'].value_counts())

# 2. Check the distribution of image labels 
print("\n--- Image Label Distribution (What is actually in the images?) ---")
display(df['label_image'].value_counts())

--- Event Distribution (What kind of disasters are here?) ---


event_name
hurricane_maria         3453
hurricane_irma          3390
hurricane_harvey        3337
california_wildfires    1156
mexico_earthquake       1027
srilanka_floods          788
iraq_iran_earthquake     457
Name: count, dtype: int64


--- Image Label Distribution (What is actually in the images?) ---


label_image
not_humanitarian                          6565
infrastructure_and_utility_damage         2761
other_relevant_information                1860
rescue_volunteering_or_donation_effort    1694
affected_individuals                       397
vehicle_damage                             230
injured_or_dead_people                      92
missing_or_found_people                      9
Name: count, dtype: int64

In [3]:
# 1. Define the rigorous rules
allowed_events = ['hurricane_maria', 'hurricane_irma', 'hurricane_harvey', 'srilanka_floods']
discarded_labels = ['not_humanitarian', 'other_relevant_information']

# 2. Apply the filters to the dataframe
# Rule A: Keep only allowed water-related events
df_filtered = df[df['event_name'].isin(allowed_events)]

# Rule B: Drop completely useless labels
df_filtered = df_filtered[~df_filtered['label_image'].isin(discarded_labels)]

print(f"[+] Original Data Count: {len(df)}")
print(f"[+] Gold Standard Data Count After Filtration: {len(df_filtered)}")
print("\n[+] Remaining Valid Image Labels for DRISHTI-Bn:")
display(df_filtered['label_image'].value_counts())

[+] Original Data Count: 13608
[+] Gold Standard Data Count After Filtration: 3755

[+] Remaining Valid Image Labels for DRISHTI-Bn:


label_image
infrastructure_and_utility_damage         2079
rescue_volunteering_or_donation_effort    1185
affected_individuals                       306
vehicle_damage                             161
injured_or_dead_people                      21
missing_or_found_people                      3
Name: count, dtype: int64

In [4]:
import os
import shutil
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# 1. Label Mapping Strategy (Macro-Classes)
label_mapping = {
    'infrastructure_and_utility_damage': 'Severe_Damage',
    'vehicle_damage': 'Severe_Damage',
    'rescue_volunteering_or_donation_effort': 'Humanitarian_Rescue',
    'affected_individuals': 'Affected_People',
    'injured_or_dead_people': 'Affected_People',
    'missing_or_found_people': 'Affected_People'
}

# Apply mapping to create a new column
df_filtered['macro_label'] = df_filtered['label_image'].map(label_mapping)

print("--- New Macro-Class Distribution ---")
display(df_filtered['macro_label'].value_counts())

# 2. Physical File Migration
RAW_DIR = '../data/raw/CrisisMMD_v2.0/'
PROCESSED_DIR = '../data/processed/'

print("\n[+] Starting physical file migration to data/processed/ ...")
successful_copies = 0
missing_files = 0

# Loop through the filtered dataframe and copy physical files
for index, row in tqdm(df_filtered.iterrows(), total=len(df_filtered)):
    # Original image path from TSV
    img_rel_path = row['image']
    src_path = os.path.join(RAW_DIR, img_rel_path)
    
    # Destination folder based on macro_label
    macro_label = row['macro_label']
    dest_folder = os.path.join(PROCESSED_DIR, macro_label)
    os.makedirs(dest_folder, exist_ok=True)
    
    # Extract filename and create dest path
    filename = os.path.basename(img_rel_path)
    dest_path = os.path.join(dest_folder, filename)
    
    # Copy file if source exists
    if os.path.exists(src_path):
        shutil.copy2(src_path, dest_path)
        successful_copies += 1
    else:
        missing_files += 1

print(f"\n[+] Migration Complete!")
print(f"[+] Successfully copied: {successful_copies} images")
if missing_files > 0:
    print(f"[-] Missing files (not found in raw directory): {missing_files}")

--- New Macro-Class Distribution ---


macro_label
Severe_Damage          2240
Humanitarian_Rescue    1185
Affected_People         330
Name: count, dtype: int64


[+] Starting physical file migration to data/processed/ ...


  0%|          | 0/3755 [00:00<?, ?it/s]

  1%|▏         | 50/3755 [00:00<00:07, 486.36it/s]

...

 96%|█████████▋| 3615/3755 [00:07<00:00, 507.71it/s]

 98%|█████████▊| 3667/3755 [00:07<00:00, 510.19it/s]

 99%|█████████▉| 3723/3755 [00:08<00:00, 520.45it/s]

100%|██████████| 3755/3755 [00:08<00:00, 459.42it/s]


[+] Migration Complete!
[+] Successfully copied: 3755 images


In [5]:
import pandas as pd
import os

PROCESSED_DIR = '../data/processed/'

# 1. Create the Master Dataset Structure
master_data = []

for index, row in df_filtered.iterrows():
    img_rel_path = row['image']
    filename = os.path.basename(img_rel_path)
    macro_label = row['macro_label']
    eng_text = row['tweet_text']
    
    # Define the exact relative path inside data/processed/
    processed_img_path = f"{macro_label}/{filename}"
    
    master_data.append({
        'image_path': processed_img_path,
        'macro_label': macro_label,
        'english_caption': eng_text
    })

# 2. Save to CSV
df_master = pd.DataFrame(master_data)
master_csv_path = os.path.join(PROCESSED_DIR, 'master_dataset.csv')
df_master.to_csv(master_csv_path, index=False)

print(f"[+] Master mapping completed! File saved at: {master_csv_path}")
print(f"[+] Total rows locked: {len(df_master)}")
print("\n--- Preview of Master Dataset ---")
display(df_master.head())

[+] Master mapping completed! File saved at: ../data/processed/master_dataset.csv
[+] Total rows locked: 3755

--- Preview of Master Dataset ---


,image_path,macro_label,english_caption
0,Humanitarian_Rescue/869972354004393987_0.jpg,Humanitarian_Rescue,Pak Navy continues Humanitarian Assistance and...
1,Humanitarian_Rescue/869972354004393987_1.jpg,Humanitarian_Rescue,Pak Navy continues Humanitarian Assistance and...
2,Humanitarian_Rescue/869972354004393987_2.jpg,Humanitarian_Rescue,Pak Navy continues Humanitarian Assistance and...
3,Humanitarian_Rescue/869972354004393987_3.jpg,Humanitarian_Rescue,Pak Navy continues Humanitarian Assistance and...
4,Severe_Damage/870035816990425089_0.jpg,Severe_Damage,After cyclone #mora #RCY Rangamati is in actio...


In [ ]:
import pandas as pd
import torch
from transformers import MarianMTModel, MarianTokenizer
from tqdm import tqdm
import os

# 1. Setup paths
PROCESSED_DIR = '../data/processed/'
master_csv_path = os.path.join(PROCESSED_DIR, 'master_dataset.csv')
output_csv_path = os.path.join(PROCESSED_DIR, 'master_dataset_translated.csv')

df = pd.read_csv(master_csv_path)

# 2. Hardware configuration (Auto-detecting GPU/CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[+] Utilizing Hardware for Inference: {device}")

# 3. Load Helsinki-NLP Model (Offline Engine)
model_name = "monirbishal/en-bn-nmt"
print(f"[+] Loading Neural Machine Translation Model: {model_name} ...")
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name).to(device)

# 4. Batch Translation Architecture
def translate_batch(texts, batch_size=16):
    translated_texts = []
    # Using tqdm for a live progress bar
    for i in tqdm(range(0, len(texts), batch_size), desc="Translating Batches"):
        batch = texts[i:i+batch_size]
        
        # Tokenize the English text
        encoded = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        
        # Generate Bengali tokens (No gradient calculation needed for inference)
        with torch.no_grad():
            generated_tokens = model.generate(**encoded)
        
        # Decode tokens back to Bengali text
        decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        translated_texts.extend(decoded)
        
    return translated_texts

# 5. Execute Translation
english_texts = df['english_caption'].tolist()
print(f"\n[+] Starting offline translation pipeline for {len(english_texts)} captions...")

# Running the translation engine
bengali_texts = translate_batch(english_texts, batch_size=16)

# Add the new Bengali captions to our dataframe
df['bengali_caption'] = bengali_texts

# 6. Save the final "Gold Standard" Dataset
df.to_csv(output_csv_path, index=False)
print(f"\n[+] Linguistic Adaptation Complete! Master file saved at: {output_csv_path}")

# Preview the final result
display(df[['english_caption', 'bengali_caption']].head())

In [ ]:
import sys
import os
import torch

# ১. লোকাল পাথ ইনজেকশন (যাতে notebooks থেকে src ফোল্ডার অ্যাক্সেস করা যায়)
sys.path.append(os.path.abspath('../'))
from src.data_loader import get_dataloaders

# ২. সুনির্দিষ্ট পাথ ও কনফিগারেশন ডিফাইন করা
CSV_PATH = '../data/processed/master_dataset_translated.csv'
IMG_DIR = '../data/processed/' # ইমেজ পাথগুলো macro_label/filename আকারে আছে

print("[+] Initializing DRISHTI-Bn DataLoaders...")

try:
    # আমরা টেস্ট করার জন্য ছোট ব্যাচ সাইজ (Batch Size = 4) ব্যবহার করব
    train_loader, val_loader = get_dataloaders(CSV_PATH, IMG_DIR, batch_size=4)
    print(f"[+] Loaders Ready. Train Batches: {len(train_loader)}, Val Batches: {len(val_loader)}")
    
    # ৩. ডেটালোডার থেকে একদম প্রথম ব্যাচটি টেনে বের করা (The Extraction)
    first_batch = next(iter(train_loader))
    
    print("\n--- 📊 Tensor Shape Verification (The Strict Baseline) ---")
    print(f"Pixel Values (Images) Shape : {first_batch['pixel_values'].shape} -> [Expected: torch.Size([4, 3, 224, 224])]")
    print(f"Input IDs (Text Tokens) Shape: {first_batch['input_ids'].shape} -> [Expected: torch.Size([4, 128])]")
    print(f"Attention Mask Shape        : {first_batch['attention_mask'].shape} -> [Expected: torch.Size([4, 128])]")
    print(f"Labels Shape                : {first_batch['labels'].shape} -> [Expected: torch.Size([4])]")
    print(f"Extracted Target Labels     : {first_batch['labels'].tolist()}")
    
    print("\n[+] DIAGNOSTIC RESULT: DataLoader Sanity Check PASSED! The architecture is production-ready.")

except Exception as e:
    print(f"\n[-] DIAGNOSTIC RESULT: Sanity Check FAILED!")
    print(f"Error Core Breakdown: {str(e)}")